# Módulo 04 · Aula 2 — Agregações e grupos

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

A aula 1 tratou de escolher linhas e colunas. Esta trata de **resumir**: transformar dez
mil pregões em oito números que respondem a uma pergunta.

É o `groupby` do módulo 02, e a estrutura mental é idêntica — dividir em grupos, calcular
algo em cada grupo, juntar o resultado. O que muda é que agora o cálculo acontece **dentro
do banco**, e só o resumo viaja até você. Com uma tabela de bilhões de linhas, essa é a
diferença entre uma resposta em dois segundos e uma máquina sem memória.

Ao final desta aula você vai:

- resumir colunas com `COUNT`, `SUM`, `AVG`, `MIN` e `MAX`;
- entender por que `COUNT(*)` e `COUNT(coluna)` dão números diferentes;
- agrupar com `GROUP BY` e filtrar grupos com `HAVING`;
- escrever `CASE WHEN` e montar uma tabela cruzada com agregação condicional;
- agrupar por ano e por mês a partir de uma data guardada como texto.

**Tempo estimado:** 75 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê o banco de dados da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "04_SQL"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

conexao = sqlite3.connect("../data/capacitacao.db")


def consultar(sql):
    return pd.read_sql_query(sql, conexao)


print("Conectado.")

## 1. As funções de agregação

Uma **função de agregação** recebe muitas linhas e devolve **um** valor.

| Função | O que faz | Em pandas |
|---|---|---|
| `COUNT(*)` | conta linhas | `len(df)` |
| `COUNT(coluna)` | conta valores **não vazios** | `df["c"].count()` |
| `SUM(coluna)` | soma | `.sum()` |
| `AVG(coluna)` | média | `.mean()` |
| `MIN` / `MAX` | menor / maior | `.min()` / `.max()` |

Sem `GROUP BY`, elas tratam a tabela inteira como um grupo só.

In [ ]:
consultar("""
    SELECT
        COUNT(*)              AS pregoes,
        MIN(data)             AS primeiro_dia,
        MAX(data)             AS ultimo_dia,
        ROUND(AVG(fechamento_ajustado), 2) AS preco_medio,
        MAX(volume)           AS maior_volume
    FROM cotacoes
    WHERE ticker = 'PETR4'
""")

Repare no `ROUND(..., 2)`. Sem ele, a média viria com quinze casas decimais.
Arredondar no banco é hábito bom: o número que chega já está no formato em que vai ser
lido.

## 2. `COUNT(*)` × `COUNT(coluna)` — o `NULL` de novo

Esta é a distinção que mais gera número errado em relatório, e ela é consequência direta da
regra da aula 1: **`COUNT(coluna)` ignora os vazios.**

In [ ]:
consultar("""
    SELECT
        COUNT(*)                     AS linhas_na_tabela,
        COUNT(idade)                 AS com_idade,
        COUNT(perfil_investidor)     AS com_perfil,
        COUNT(patrimonio_investido)  AS com_patrimonio
    FROM clientes
""")

400 linhas, mas só 379 com idade. Os 21 que faltam são os vazios que você contou
com `IS NULL` na aula 1 — e são os mesmos 21 do `.isna().sum()` do módulo 02.

A armadilha prática: se você calcular `SUM(patrimonio) / COUNT(*)` achando que fez uma
média, o resultado está errado — dividiu por 400 uma soma de 384 valores. É por isso que
`AVG` existe: ele já ignora os vazios dos dois lados da conta, e é o que você quer em
quase todo caso.

> **Cuidado com a interpretação, no entanto.** `AVG` ignorar o vazio é o comportamento
> certo *tecnicamente* e uma decisão *analítica* que você precisa assumir: você está
> dizendo que os clientes sem patrimônio registrado se parecem com os demais. Às vezes é
> verdade; às vezes o dado falta justamente porque o cliente é diferente. O módulo 02 já
> tratava disso — a ferramenta muda, a responsabilidade não.

In [ ]:
# COUNT(DISTINCT ...) conta valores distintos, não linhas
consultar("""
    SELECT
        COUNT(*)                        AS linhas,
        COUNT(DISTINCT ticker)          AS papeis,
        COUNT(DISTINCT data)            AS pregoes_distintos
    FROM cotacoes
""")

9.968 linhas, 8 papéis, 1.246 pregões — e 8 × 1.246 = 9.968. As séries estão
perfeitamente alinhadas: todo papel tem todo pregão. Essa conta de um minuto é uma boa
conferência de sanidade em qualquer base nova.

## 3. `GROUP BY`

Agora o resumo por grupo. A ideia é a mesma do `groupby` do pandas: as linhas são
separadas em baldes segundo o valor de uma coluna, e a agregação roda dentro de cada
balde.

In [ ]:
consultar("""
    SELECT
        ticker,
        COUNT(*)                            AS pregoes,
        ROUND(AVG(fechamento_ajustado), 2)  AS preco_medio,
        ROUND(MIN(fechamento_ajustado), 2)  AS minimo,
        ROUND(MAX(fechamento_ajustado), 2)  AS maximo
    FROM cotacoes
    GROUP BY ticker
    ORDER BY preco_medio DESC
""")

### A regra de ouro do `GROUP BY`

> **Toda coluna do `SELECT` tem que estar no `GROUP BY` ou dentro de uma função de
> agregação.**

A razão é lógica, não burocrática. Se você agrupa por `ticker`, cada linha do resultado
representa 1.246 pregões. Perguntar "qual é a `data` desse grupo?" não tem resposta — são
1.246 datas diferentes. Ou você agrega (`MIN(data)`, `MAX(data)`) ou você agrupa também
por data.

**E aqui o SQLite arma uma cilada.** Bancos sérios recusam a consulta abaixo com erro. O
SQLite aceita e devolve um valor **arbitrário** — sem aviso, sem erro, com cara de
resposta certa:

In [ ]:
consultar("""
    SELECT ticker, data, fechamento
    FROM cotacoes
    GROUP BY ticker
""")

A coluna `data` aí é uma linha qualquer de cada grupo. Não é a primeira, não é a
última, não é nada — é o que o banco tinha à mão.

**No PostgreSQL isso é um erro de execução**, e é assim que deveria ser. Trate como erro
mesmo aqui: se uma coluna não está no `GROUP BY` nem dentro de uma agregação, a consulta
está errada, ainda que o SQLite responda.

In [ ]:
# Agrupando por mais de uma coluna: o grupo é a COMBINAÇÃO
consultar("""
    SELECT
        setor,
        controle,
        COUNT(*)          AS empresas,
        MIN(ano_fundacao) AS mais_antiga
    FROM empresas
    GROUP BY setor, controle
    ORDER BY setor, controle
""")

## 4. `HAVING`: filtrar **grupos**

Esta é a distinção clássica de SQL, e ela decorre da ordem de execução da aula 1.

| Cláusula | Filtra | Quando roda |
|---|---|---|
| `WHERE` | **linhas**, antes de agrupar | antes do `GROUP BY` |
| `HAVING` | **grupos**, depois de agregar | depois do `GROUP BY` |

A ordem completa passa a ser:

```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

Um `WHERE` não consegue enxergar `COUNT(*)`, porque quando ele roda ainda não existe
grupo nenhum para contar. Para filtrar por um resultado de agregação, só `HAVING`.

In [ ]:
# Quais setores têm mais de uma empresa na nossa amostra?
consultar("""
    SELECT
        setor,
        COUNT(*) AS empresas
    FROM empresas
    GROUP BY setor
    HAVING COUNT(*) > 1
    ORDER BY empresas DESC
""")

### Os dois juntos

Usar `WHERE` e `HAVING` na mesma consulta é comum, e cada um faz um trabalho diferente:
o `WHERE` escolhe **quais pregões entram na conta**, o `HAVING` escolhe **quais papéis
aparecem no resultado**.

In [ ]:
consultar("""
    SELECT
        ticker,
        COUNT(*)                            AS pregoes_em_2025,
        ROUND(AVG(fechamento_ajustado), 2)  AS preco_medio_2025
    FROM cotacoes
    WHERE data >= '2025-01-01'            -- filtra LINHAS: só 2025
    GROUP BY ticker
    HAVING AVG(fechamento_ajustado) > 30  -- filtra GRUPOS: só os caros
    ORDER BY preco_medio_2025 DESC
""")

> **Desempenho.** Sempre que a mesma restrição puder ser escrita nos dois lugares,
> escreva no `WHERE`. Ele joga linhas fora *antes* do trabalho de agrupar; o `HAVING`
> descarta depois que o trabalho já foi feito. Em tabela grande, a diferença é enorme.

## 5. `CASE WHEN`: o `if` do SQL

`CASE` classifica cada linha, e é a peça que falta para o SQL responder perguntas de
negócio de verdade.

```sql
CASE
    WHEN condição1 THEN valor1
    WHEN condição2 THEN valor2
    ELSE            valor_padrão
END
```

As condições são testadas em ordem, e a primeira que der verdadeiro vence — igual a uma
cadeia `if`/`elif`/`else` do módulo 01. Sem `ELSE`, o que não casar vira `NULL`.

In [ ]:
consultar("""
    SELECT
        ticker,
        empresa,
        ano_fundacao,
        CASE
            WHEN ano_fundacao < 1950 THEN 'Centenária'
            WHEN ano_fundacao < 1990 THEN 'Madura'
            ELSE                          'Recente'
        END AS geracao
    FROM empresas
    ORDER BY ano_fundacao
""")

E `CASE` combina com `GROUP BY`: dá para agrupar por uma categoria que **não
existe na tabela** e é inventada na hora.

In [ ]:
consultar("""
    SELECT
        CASE
            WHEN idade IS NULL   THEN 'sem informação'
            WHEN idade <= 0 OR idade > 120 THEN 'valor impossível'
            WHEN idade < 30      THEN 'até 29'
            WHEN idade < 50      THEN '30 a 49'
            ELSE                      '50 ou mais'
        END AS faixa_etaria,
        COUNT(*) AS clientes
    FROM clientes
    GROUP BY faixa_etaria
    ORDER BY clientes DESC
""")

Repare que colocamos a checagem de `IS NULL` e a de valor impossível **antes**
das faixas normais. A ordem importa: como as condições são testadas de cima para baixo, se
`idade < 30` viesse primeiro, a idade `-3` cairia em "até 29" e o defeito da base sumiria
dentro de uma categoria de aparência inocente.

E repare no `<=` do valor impossível. Com `idade < 0` — que é o que a maioria das pessoas
escreve — os clientes de idade **zero** escapam da rede e são contados como "até 29". A
base tem dois deles. Um sinal de igual esquecido move dois clientes de "dado quebrado"
para "jovem", e nada no resultado denuncia isso.

É a mesma armadilha da aula 2 do módulo 01 — a ordem das condições em `if`/`elif` muda o
resultado —, agora com consequência sobre um número que alguém vai levar para uma
reunião.

## 6. Agregação condicional: a tabela cruzada

Combinar `SUM` com `CASE` produz o que o pandas faz com `pivot_table`: uma tabela onde
cada coluna é uma categoria.

O truque é sempre o mesmo — `CASE` devolve 1 quando a linha pertence à coluna e 0 quando
não pertence; `SUM` conta os uns.

In [ ]:
consultar("""
    SELECT
        setor,
        COUNT(*)                                                  AS total,
        SUM(CASE WHEN controle = 'Estatal' THEN 1 ELSE 0 END)     AS estatais,
        SUM(CASE WHEN controle = 'Privada' THEN 1 ELSE 0 END)     AS privadas
    FROM empresas
    GROUP BY setor
    ORDER BY total DESC
""")

A mesma técnica com `AVG` em vez de `SUM` dá a média de um subconjunto por
grupo. Aqui, o preço médio de cada papel em dois anos diferentes, lado a lado:

In [ ]:
consultar("""
    SELECT
        ticker,
        ROUND(AVG(CASE WHEN data < '2022-01-01' THEN fechamento_ajustado END), 2) AS media_2021,
        ROUND(AVG(CASE WHEN data >= '2025-01-01' THEN fechamento_ajustado END), 2) AS media_2025
    FROM cotacoes
    GROUP BY ticker
    ORDER BY ticker
""")

Repare que aqui o `CASE` **não tem `ELSE`**, e isso é proposital: o que não casa
vira `NULL`, e `AVG` ignora `NULL`. Se tivéssemos escrito `ELSE 0`, os zeros entrariam na
média e a achatariam — um erro que passa despercebido porque o resultado continua sendo um
número plausível.

> **A regra:** com `SUM` contando ocorrências, use `ELSE 0`. Com `AVG`, **nunca** — deixe
> virar `NULL`.

## 7. Datas: agrupar por ano e por mês

O SQLite guarda data como texto, então extrair o ano é extrair um pedaço do texto. A
função é `strftime`, e os padrões são os mesmos do Python: `%Y` para ano, `%m` para mês.

In [ ]:
consultar("""
    SELECT
        strftime('%Y', data)                AS ano,
        COUNT(*)                            AS pregoes,
        ROUND(AVG(fechamento_ajustado), 2)  AS preco_medio,
        ROUND(MAX(maxima), 2)               AS maxima_do_ano
    FROM cotacoes
    WHERE ticker = 'VALE3'
    GROUP BY ano
    ORDER BY ano
""")

In [ ]:
# Ano-mês, o formato que a aula de EDA do módulo 03 usava
consultar("""
    SELECT
        strftime('%Y-%m', data)             AS ano_mes,
        ROUND(AVG(fechamento_ajustado), 2)  AS preco_medio,
        SUM(volume)                         AS volume_total
    FROM cotacoes
    WHERE ticker = 'ITUB4'
      AND data >= '2025-07-01'
    GROUP BY ano_mes
    ORDER BY ano_mes
""")

> **`strftime` é do SQLite.** No PostgreSQL seria `TO_CHAR(data, 'YYYY-MM')` ou
> `DATE_TRUNC('month', data)`; no BigQuery, `FORMAT_DATE`. A ideia é universal, o nome da
> função não é. Funções de data são a parte do SQL que menos se parece entre bancos — é a
> primeira coisa a procurar na documentação quando você trocar de ambiente.

## 8. Conferindo contra o pandas

De novo o teste que fecha a aula: a mesma pergunta, pelos dois caminhos.

> *Qual o retorno de cada papel de 2021 a 2025, medido do primeiro ao último fechamento
> ajustado?*

Isso exige o primeiro e o último valor de cada grupo — e é aqui que aparece uma diferença
real entre as duas ferramentas. `MIN(data)` dá a data mais antiga, mas não o preço
**naquela** data. Por enquanto, resolvemos com agregação condicional; a aula 4 vai mostrar
a ferramenta certa para isso (`FIRST_VALUE` e janelas).

In [ ]:
via_sql = consultar("""
    SELECT
        ticker,
        ROUND(MIN(CASE WHEN data = '2021-01-04' THEN fechamento_ajustado END), 2) AS inicio,
        ROUND(MIN(CASE WHEN data = '2025-12-30' THEN fechamento_ajustado END), 2) AS fim
    FROM cotacoes
    GROUP BY ticker
    ORDER BY ticker
""")
via_sql["retorno_pct"] = ((via_sql["fim"] / via_sql["inicio"] - 1) * 100).round(1)
via_sql

In [ ]:
acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"]).sort_values(["ticker", "data"])

via_pandas = (
    acoes.groupby("ticker")["fechamento_ajustado"]
    .agg(inicio="first", fim="last")
    .round(2)
    .reset_index()
)
via_pandas["retorno_pct"] = ((via_pandas["fim"] / via_pandas["inicio"] - 1) * 100).round(1)
via_pandas

In [ ]:
import numpy as np

print("Mesmo resultado:",
      np.array_equal(via_sql["retorno_pct"].values, via_pandas["retorno_pct"].values))

Bateu — e vale notar por quê: só bateu porque **todos os papéis têm exatamente os
mesmos 1.246 pregões**, de forma que `'2021-01-04'` é de fato o primeiro dia de cada série.
Em uma base onde os papéis começam em datas diferentes, essa consulta daria `NULL` para
alguns e o `first` do pandas continuaria certo.

É um bom exemplo de consulta que funciona por causa de uma propriedade dos dados, não da
lógica. Isso é frágil — e é exatamente o que as funções de janela da aula 4 resolvem.

## 9. O mapa, atualizado

| Pergunta | pandas | SQL |
|---|---|---|
| quais colunas | `[[...]]` | `SELECT` |
| de onde | `df` | `FROM` |
| filtrar linhas | máscara booleana | `WHERE` |
| agrupar | `.groupby()` | `GROUP BY` |
| resumir | `.agg()`, `.mean()`, `.sum()` | `AVG()`, `SUM()`, `COUNT()` |
| filtrar grupos | `.filter()` depois do groupby | `HAVING` |
| classificar em faixas | `np.select`, `pd.cut` | `CASE WHEN` |
| tabela cruzada | `pivot_table` | `SUM(CASE WHEN ...)` |
| ordenar | `.sort_values()` | `ORDER BY` |
| primeiras n | `.head(n)` | `LIMIT n` |

## 10. Recapitulando

- Funções de agregação recebem muitas linhas e devolvem uma. Sem `GROUP BY`, a tabela
  inteira é um grupo só.
- **`COUNT(*)` conta linhas; `COUNT(coluna)` ignora os vazios.** Confundir os dois é a
  origem mais comum de número errado em relatório.
- No `GROUP BY`, toda coluna do `SELECT` tem que estar agrupada ou agregada. O SQLite
  deixa passar e devolve lixo silencioso — trate como erro mesmo assim.
- **`WHERE` filtra linhas antes de agrupar; `HAVING` filtra grupos depois de agregar.**
  Quando der para usar os dois, prefira o `WHERE`.
- `CASE WHEN` classifica linha a linha, e a ordem das condições muda o resultado. Trate os
  vazios e os valores impossíveis **primeiro**.
- `SUM(CASE ...)` monta tabela cruzada. Com `SUM`, use `ELSE 0`; com `AVG`, nunca.
- `strftime('%Y', data)` extrai o ano — no SQLite. Funções de data são o que menos se
  parece entre bancos.

**Próxima aula:** `JOIN` — juntar de volta as tabelas que a normalização separou.

In [ ]:
conexao.close()
print("Conexão fechada.")